# Advanced SQL Commands

This notebook covers commonly used advanced SQL commands with explanations and examples.

## Common Table Expressions (CTE)
CTEs define temporary result sets that can be referenced within a query.

```sql
WITH department_avg AS (
    SELECT department, AVG(salary) AS avg_sal
    FROM employees
    GROUP BY department
)
SELECT e.name, e.salary
FROM employees e
JOIN department_avg d ON e.department = d.department
WHERE e.salary > d.avg_sal;
```

## Recursive CTE
Recursive CTEs reference themselves to traverse hierarchical data.

```sql
WITH RECURSIVE manager_path AS (
    SELECT id, manager_id, name, 1 AS level
    FROM employees
    WHERE manager_id IS NULL
  UNION ALL
    SELECT e.id, e.manager_id, e.name, mp.level + 1
    FROM employees e
    JOIN manager_path mp ON e.manager_id = mp.id
)
SELECT * FROM manager_path;
```

## Window Functions
Window functions perform calculations across sets of rows related to the current row.

```sql
SELECT name,
       salary,
       ROW_NUMBER() OVER (ORDER BY salary DESC) AS row_num,
       LAG(salary) OVER (ORDER BY salary) AS previous_salary
FROM employees;
```

## Correlated Subquery
A correlated subquery uses values from the outer query for its comparisons.

```sql
SELECT name, salary
FROM employees e
WHERE salary > (
    SELECT AVG(salary)
    FROM employees
    WHERE department = e.department
);
```

## EXISTS and NOT EXISTS
`EXISTS` tests for the presence of rows returned by a subquery, while `NOT EXISTS` checks for absence.

```sql
SELECT name
FROM employees e
WHERE EXISTS (
    SELECT 1 FROM bonuses b
    WHERE b.employee_id = e.id
);

SELECT name
FROM employees e
WHERE NOT EXISTS (
    SELECT 1 FROM bonuses b
    WHERE b.employee_id = e.id
);
```

## PIVOT
PIVOT rotates rows into columns, making cross-tab reports easier.

```sql
SELECT *
FROM (
    SELECT department, quarter, revenue
    FROM sales
) AS src
PIVOT (
    SUM(revenue)
    FOR quarter IN ('Q1', 'Q2', 'Q3', 'Q4')
) AS p;
```

## MERGE
MERGE performs insert, update, or delete operations in a single statement based on matched records.

```sql
MERGE INTO target t
USING source s
ON t.id = s.id
WHEN MATCHED THEN
  UPDATE SET t.value = s.value
WHEN NOT MATCHED THEN
  INSERT (id, value) VALUES (s.id, s.value);
```